# Part 3 — Machine Learning: Predict No-Shows and Choose the Model Honestly

## 3.1 — Features, Train/Test Split and Baseline

In this section, the cleaned appointment dataset is loaded and prepared for machine learning.

The features are selected according to the project requirements, categorical variables are encoded, and the data is divided into training and testing sets using an 80/20 split.

A baseline model is also created by predicting that every appointment will be attended. This provides a reference point for evaluating the machine learning models.

In [ ]:
import pandas as pd
import numpy as np

# Load the cleaned dataset from Part 1
df = pd.read_csv('/content/appointments_clean.csv')

# Check the dataset
print("Dataset shape:", df.shape)
print("\nFirst five rows:")
display(df.head())

Dataset shape: (990, 16)

First five rows:


,appointment_id,patient_id,clinic_id,booked_on,appt_date,age,gender,distance_km,fee_inr,booking_channel,visit_type,reminder_sent,prior_no_shows,status,lead_days,no_show
0,A0001,P509,C05,2025-01-24,2025-01-25,60,F,18.0,850,Phone,Consultation,Yes,1,No-Show,1,1
1,A0002,P046,C07,2025-05-27,2025-06-06,39,M,12.5,350,Walk-In,Follow-up,No,2,Attended,10,0
2,A0003,P009,C02,2025-03-24,2025-03-30,63,F,4.1,850,Walk-In,Consultation,No,0,Attended,6,0
3,A0004,P482,C01,2025-04-18,2025-04-22,52,F,7.5,750,App,Consultation,Yes,0,Attended,4,0
4,A0005,P251,C01,2024-11-23,2025-01-02,64,F,19.0,700,Phone,Consultation,No,0,No-Show,40,1


In [ ]:
# Keep appointment IDs separately for later
appointment_ids = df['appointment_id'].copy()

# Select the features
X = df[
    [
        'age',
        'distance_km',
        'fee_inr',
        'lead_days',
        'prior_no_shows',
        'reminder_sent',
        'booking_channel',
        'visit_type'
    ]
].copy()

# Convert reminder_sent into numbers
X['reminder_sent'] = X['reminder_sent'].map({
    'Yes': 1,
    'No': 0
})

# Convert categorical columns into numerical columns
X = pd.get_dummies(
    X,
    columns=['booking_channel', 'visit_type']
)

# Target variable
y = df['no_show']

print("Feature shape:", X.shape)
print("\nFeature names:")
print(X.columns.tolist())

Feature shape: (990, 11)

Feature names:
['age', 'distance_km', 'fee_inr', 'lead_days', 'prior_no_shows', 'reminder_sent', 'booking_channel_App', 'booking_channel_Phone', 'booking_channel_Walk-In', 'visit_type_Consultation', 'visit_type_Follow-up']


In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))
print("No-shows in test set:", int(y_test.sum()))

Training rows: 792
Testing rows: 198
No-shows in test set: 49


In [ ]:
from sklearn.metrics import accuracy_score

# Always predict 0 (Attended)
baseline_predictions = np.zeros(len(y_test))

baseline_accuracy = accuracy_score(
    y_test,
    baseline_predictions
)

print("Baseline accuracy:", round(baseline_accuracy, 4))

Baseline accuracy: 0.7525


## 3.2 — Logistic Regression

In this section, Logistic Regression is used as the first machine learning model to predict appointment no-shows.

The features are standardised using StandardScaler, which is fitted only on the training data to avoid data leakage.

The model is evaluated using accuracy, confusion matrix, precision, recall, F1-score and ROC-AUC. Feature coefficients are also examined to understand how different variables relate to no-show predictions.

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

# Create the scaler
scaler = StandardScaler()

# Fit on training data and transform it
X_train_s = scaler.fit_transform(X_train)

# Transform test data using the same scaler
X_test_s = scaler.transform(X_test)

print("Training data shape:", X_train_s.shape)
print("Testing data shape:", X_test_s.shape)

Training data shape: (792, 11)
Testing data shape: (198, 11)


In [ ]:
# Create the model
model = LogisticRegression(max_iter=1000)

# Train the model
model.fit(X_train_s, y_train)

# Predict test results
y_pred = model.predict(X_test_s)

# Get probability of no-show
y_prob = model.predict_proba(X_test_s)[:, 1]

print("Model training completed!")

Model training completed!


In [ ]:
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

accuracy = accuracy_score(y_test, y_pred)

cm = confusion_matrix(y_test, y_pred)

precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_prob)

print("Accuracy:", round(accuracy, 3))
print("\nConfusion Matrix:")
print(cm)

print("\nPrecision:", round(precision, 3))
print("Recall:", round(recall, 3))
print("F1 Score:", round(f1, 3))
print("ROC-AUC:", round(roc_auc, 3))

Accuracy: 0.788

Confusion Matrix:
[[139  10]
 [ 32  17]]

Precision: 0.63
Recall: 0.347
F1 Score: 0.447
ROC-AUC: 0.789


In [ ]:
coefficients = pd.DataFrame({
    'feature': X.columns,
    'coefficient': model.coef_[0]
})

coefficients = coefficients.sort_values(
    'coefficient',
    ascending=True
)

display(coefficients)

,feature,coefficient
5,reminder_sent,-0.580669
0,age,-0.253880
8,booking_channel_Walk-In,-0.186364
6,booking_channel_App,-0.046804
10,visit_type_Follow-up,-0.015658
9,visit_type_Consultation,0.015658
7,booking_channel_Phone,0.215881
1,distance_km,0.253448
2,fee_inr,0.452761
3,lead_days,0.614471


In [ ]:
# Find reminder_sent column position
reminder_idx = list(X.columns).index('reminder_sent')

# Get its model coefficient
reminder_coef = model.coef_[0][reminder_idx]

# Exponential of the scaled coefficient
scaled_odds_ratio = np.exp(reminder_coef)

# Adjust for the feature's standard deviation
actual_odds_ratio = np.exp(
    reminder_coef / scaler.scale_[reminder_idx]
)

print("Scaled coefficient:", round(reminder_coef, 3))
print("Exp(coefficient):", round(scaled_odds_ratio, 3))
print("Reminder No-to-Yes odds ratio:", round(actual_odds_ratio, 3))

Scaled coefficient: -0.581
Exp(coefficient): 0.56
Reminder No-to-Yes odds ratio: 0.309


## 3.3 — Decision Tree

In this section, Decision Tree classifiers are used to predict appointment no-shows.

Two models are evaluated:
- A full-depth Decision Tree.
- A restricted Decision Tree with a maximum depth of 3.

Training and testing accuracy are compared to understand how model complexity affects performance and to identify potential overfitting.

In [ ]:
from sklearn.tree import DecisionTreeClassifier

print("Decision Tree library imported successfully!")

Decision Tree library imported successfully!


In [ ]:
# Create a full-depth decision tree
tree_full = DecisionTreeClassifier(random_state=42)

# Train the model
tree_full.fit(X_train, y_train)

# Predictions on training and testing data
train_pred_full = tree_full.predict(X_train)
test_pred_full = tree_full.predict(X_test)

# Calculate accuracy
train_acc_full = accuracy_score(y_train, train_pred_full)
test_acc_full = accuracy_score(y_test, test_pred_full)

print("Full-depth Decision Tree")
print("Training accuracy:", round(train_acc_full, 3))
print("Testing accuracy:", round(test_acc_full, 3))
print("Tree depth:", tree_full.get_depth())

Full-depth Decision Tree
Training accuracy: 1.0
Testing accuracy: 0.672
Tree depth: 15


In [ ]:
# Create a restricted decision tree
tree_3 = DecisionTreeClassifier(
    max_depth=3,
    random_state=42
)

# Train the model
tree_3.fit(X_train, y_train)

# Predictions
train_pred_3 = tree_3.predict(X_train)
test_pred_3 = tree_3.predict(X_test)

# Calculate accuracy
train_acc_3 = accuracy_score(y_train, train_pred_3)
test_acc_3 = accuracy_score(y_test, test_pred_3)

print("Depth-3 Decision Tree")
print("Training accuracy:", round(train_acc_3, 3))
print("Testing accuracy:", round(test_acc_3, 3))
print("Tree depth:", tree_3.get_depth())

Depth-3 Decision Tree
Training accuracy: 0.822
Testing accuracy: 0.778
Tree depth: 3


## 3.4 — Random Forest

In this section, a Random Forest classifier is trained using 100 decision trees.

The model is evaluated using accuracy, recall and ROC-AUC. Training and testing performance are also compared to assess generalisation.

Feature importance scores are examined to identify the three most influential features in predicting appointment no-shows.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

# Create Random Forest model
rf = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

# Train the model
rf.fit(X_train, y_train)

# Predictions
rf_train_pred = rf.predict(X_train)
rf_test_pred = rf.predict(X_test)

# Probabilities for ROC-AUC
rf_test_prob = rf.predict_proba(X_test)[:, 1]

print("Random Forest training completed!")
print("Training accuracy:", round(
    accuracy_score(y_train, rf_train_pred), 3
))

Random Forest training completed!
Training accuracy: 1.0


In [ ]:
rf_accuracy = accuracy_score(y_test, rf_test_pred)
rf_recall = recall_score(y_test, rf_test_pred)
rf_auc = roc_auc_score(y_test, rf_test_prob)

print("Random Forest Results")
print("Accuracy:", round(rf_accuracy, 3))
print("Recall:", round(rf_recall, 3))
print("ROC-AUC:", round(rf_auc, 3))

Random Forest Results
Accuracy: 0.747
Recall: 0.306
ROC-AUC: 0.706


In [ ]:
feature_importance = pd.DataFrame({
    'feature': X.columns,
    'importance': rf.feature_importances_
})

feature_importance = feature_importance.sort_values(
    'importance',
    ascending=False
)

print("Top 3 important features:")
display(feature_importance.head(3))

Top 3 important features:


,feature,importance
3,lead_days,0.219905
0,age,0.196788
1,distance_km,0.194739


## 3.5 — Cross-Validation, Model Selection and Threshold Tuning

In this section, the models are compared using 5-fold stratified cross-validation and ROC-AUC.

Random Forest hyperparameters are also tuned using GridSearchCV. The best-performing model is selected based on cross-validation results.

Out-of-fold predicted probabilities are then used to compare classification thresholds, including 0.50 and 0.35. The selected threshold is evaluated using precision, recall and F1-score.

Finally, the selected model is fitted on the training data and evaluated on the test data. The resulting appointment risk scores are exported for the next stage of the project.

In [ ]:
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
# Five-fold cross-validation
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

# Define the three models
models = {
    'Logistic Regression': Pipeline([
        ('scaler', StandardScaler()),
        ('model', LogisticRegression(max_iter=1000))
    ]),

    'Decision Tree': DecisionTreeClassifier(
        max_depth=3,
        random_state=42
    ),

    'Random Forest': RandomForestClassifier(
        n_estimators=100,
        random_state=42
    )
}

# Compare using ROC-AUC
cv_results = []

for name, model_cv in models.items():

    scores = cross_val_score(
        model_cv,
        X_train,
        y_train,
        cv=cv,
        scoring='roc_auc'
    )

    cv_results.append({
        'Model': name,
        'Mean ROC-AUC': scores.mean(),
        'Std ROC-AUC': scores.std()
    })

cv_table = pd.DataFrame(cv_results)

display(cv_table.round(3))

,Model,Mean ROC-AUC,Std ROC-AUC
0,Logistic Regression,0.777,0.012
1,Decision Tree,0.661,0.012
2,Random Forest,0.749,0.040


In [ ]:
from sklearn.model_selection import GridSearchCV

rf_grid = GridSearchCV(
    RandomForestClassifier(random_state=42),
    {
        "n_estimators": [100, 200],
        "max_depth": [3, 5, 8]
    },
    cv=cv,
    scoring="roc_auc"
)

rf_grid.fit(X_train, y_train)

print("Best parameters:", rf_grid.best_params_)
print("Best cross-validation ROC-AUC:", round(rf_grid.best_score_, 3))

Best parameters: {'max_depth': 5, 'n_estimators': 200}
Best cross-validation ROC-AUC: 0.761


In [ ]:
from sklearn.model_selection import cross_val_score

lr_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=1000))
])

models_for_selection = {
    "Logistic Regression": lr_pipe,
    "Decision Tree": DecisionTreeClassifier(
        max_depth=3,
        random_state=42
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=100,
        random_state=42
    )
}

selection_results = []

for name, candidate in models_for_selection.items():
    score = cross_val_score(
        candidate,
        X_train,
        y_train,
        cv=cv,
        scoring="roc_auc"
    ).mean()

    selection_results.append({
        "Model": name,
        "Mean CV ROC-AUC": score
    })

# Add the tuned Random Forest
selection_results.append({
    "Model": "Tuned Random Forest",
    "Mean CV ROC-AUC": rf_grid.best_score_
})

selection_table = pd.DataFrame(selection_results)

display(selection_table.round(3))

,Model,Mean CV ROC-AUC
0,Logistic Regression,0.777
1,Decision Tree,0.661
2,Random Forest,0.749
3,Tuned Random Forest,0.761


In [ ]:

from sklearn.model_selection import cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import precision_score, recall_score, f1_score

# Logistic Regression was selected based on cross-validation results.
# Create the final model pipeline explicitly.
final_model = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LogisticRegression(max_iter=1000))
])

# Generate out-of-fold probabilities using 5-fold cross-validation
train_prob = cross_val_predict(
    final_model,
    X_train,
    y_train,
    cv=cv,
    method='predict_proba'
)[:, 1]

print("OOF probabilities generated successfully!")
print("Number of probabilities:", len(train_prob))

# Compare the two required thresholds
threshold_results = []

for threshold in [0.50, 0.35]:

    train_pred_threshold = (
        train_prob >= threshold
    ).astype(int)

    threshold_results.append({
        'Threshold': threshold,
        'Precision': precision_score(
            y_train, train_pred_threshold, zero_division=0
        ),
        'Recall': recall_score(
            y_train, train_pred_threshold, zero_division=0
        ),
        'F1 Score': f1_score(
            y_train, train_pred_threshold, zero_division=0
        )
    })

threshold_table = pd.DataFrame(threshold_results)

display(threshold_table.round(3))

OOF probabilities generated successfully!
Number of probabilities: 792


,Threshold,Precision,Recall,F1 Score
0,0.50,0.609,0.249,0.353
1,0.35,0.534,0.467,0.498


In [ ]:
from sklearn.metrics import classification_report
# Train final model on the complete training dataset
final_model.fit(X_train, y_train)

# Get probabilities for the untouched test dataset
test_prob = final_model.predict_proba(X_test)[:, 1]

# Apply the selected threshold
selected_threshold = 0.35

test_pred = (test_prob >= selected_threshold).astype(int)

# Evaluate final model
print("Selected threshold:", selected_threshold)
print("Test Accuracy:", round(accuracy_score(y_test, test_pred), 3))
print("Test Precision:", round(precision_score(y_test, test_pred), 3))
print("Test Recall:", round(recall_score(y_test, test_pred), 3))
print("Test F1 Score:", round(f1_score(y_test, test_pred), 3))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, test_pred))

print("\nClassification Report:")
print(classification_report(y_test, test_pred))

Selected threshold: 0.35
Test Accuracy: 0.788
Test Precision: 0.571
Test Recall: 0.571
Test F1 Score: 0.571

Confusion Matrix:
[[128  21]
 [ 21  28]]

Classification Report:
              precision    recall  f1-score   support

           0       0.86      0.86      0.86       149
           1       0.57      0.57      0.57        49

    accuracy                           0.79       198
   macro avg       0.72      0.72      0.72       198
weighted avg       0.79      0.79      0.79       198



In [ ]:
from sklearn.metrics import roc_auc_score

# Calculate test ROC-AUC using probabilities
test_auc = roc_auc_score(y_test, test_prob)

print("Test ROC-AUC:", round(test_auc, 3))
print("Number of appointments flagged:", int(test_pred.sum()))

# Create the required risk scores file
test_risk_scores = pd.DataFrame({
    "appointment_id": df.loc[X_test.index, "appointment_id"].to_numpy(),
    "no_show": y_test.to_numpy(),
    "risk_prob": test_prob,
    "flag": test_pred
})

# Save CSV
test_risk_scores.to_csv(
    "/content/test_risk_scores.csv",
    index=False
)

print("\nSaved test_risk_scores.csv successfully!")
print("File shape:", test_risk_scores.shape)

display(test_risk_scores.head())

Test ROC-AUC: 0.789
Number of appointments flagged: 49

Saved test_risk_scores.csv successfully!
File shape: (198, 4)


,appointment_id,no_show,risk_prob,flag
0,A0217,0,0.021565,0
1,A0336,0,0.150917,0
2,A0508,0,0.518692,1
3,A0314,0,0.191300,0
4,A0089,0,0.167718,0
